# November Task 11
TASK 11: BASELINE COMPARISON AND FAILURE THEMES
**PURPOSE:** Reuse the existing Task 7 and Task 8 outputs. We are NOT retraining models or changing
thresholds. First, we verify that all required files are available.


In [53]:
from pathlib import Path

# Use the current PyCharm project directory.
PROJECT_ROOT = Path.cwd()

print("PROJECT DIRECTORY:")
print(PROJECT_ROOT)


PROJECT DIRECTORY:
C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks


## STEP 1
1. Locate the existing output folders

In [54]:
TASK7_DIR = PROJECT_ROOT / "october_task7_outputs"
TASK8_DIR = PROJECT_ROOT / "october_task8_outputs"
TASK4_DIR = PROJECT_ROOT / "october_task4_outputs"
CHUNKS_DIR = PROJECT_ROOT / "task12b_outputs"

### 2. Check the Task 7 baseline predictions
Task 7 actually produced keyword and logistic regression predictions. We will use those existing outputs.

In [55]:
TASK7_FILE = (
    TASK7_DIR / "task7_baseline_row_level_predictions.parquet"
)

TASK7_THRESHOLDS = (
    TASK7_DIR / "task7_baseline_thresholds_used.csv"
)

### 3. Check the Task 8 transformer thresholds
Task 8 selected weighted Legal-BERT.
Each of its three seeds has category-specific thresholds.

In [56]:
TASK8_THRESHOLDS = (
    TASK8_DIR / "task8_frozen_thresholds.csv"
)

### 4. Check the saved weighted Legal-BERT scores
We reuse the saved scores instead of running the model again.

In [57]:
WEIGHTED_DIR = TASK4_DIR / "full_weighted"

seeds = [42, 123, 2026]

print("\nWEIGHTED LEGAL-BERT SCORES:")

for seed in seeds:
    score_file = (
        WEIGHTED_DIR
        / f"weighted_Legal-BERT_seed{seed}_val_scores.npz"
    )

    print(f"Seed {seed}: {score_file.exists()}")


WEIGHTED LEGAL-BERT SCORES:
Seed 42: False
Seed 123: False
Seed 2026: False


### 5. Check the validation chunk text
We need the actual text later to inspect examples and understand why the models made certain errors.

In [58]:
VALIDATION_CHUNKS = (
    CHUNKS_DIR / "test_chunks.parquet"
)

### 6. Print all file checks

In [59]:
required_files = {
    "Task 7 baseline predictions": TASK7_FILE,
    "Task 7 thresholds": TASK7_THRESHOLDS,
    "Task 8 frozen thresholds": TASK8_THRESHOLDS,
    "Validation chunk text": VALIDATION_CHUNKS,
}

print("\nREQUIRED FILE CHECK:")

for name, path in required_files.items():
    print(f"{name}: {path.exists()}")
    print(f"  Location: {path}")

print("\nStep 1 complete. Review any False results above.")



REQUIRED FILE CHECK:
Task 7 baseline predictions: True
  Location: C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks\october_task7_outputs\task7_baseline_row_level_predictions.parquet
Task 7 thresholds: True
  Location: C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks\october_task7_outputs\task7_baseline_thresholds_used.csv
Task 8 frozen thresholds: True
  Location: C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks\october_task8_outputs\task8_frozen_thresholds.csv
Validation chunk text: True
  Location: C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks\task12b_outputs\test_chunks.parquet

Step 1 complete. Review any False results above.


## STEP 2: INSPECT TASK 7 AND LOCATE TASK 8 INPUTS
We want to confirm:
1. The actual Task 7 table structure.
2. The number of validation rows.
3. Whether masked rows are present.
4. Whether the saved transformer scores exist elsewhere.

In [60]:

import pandas as pd
from pathlib import Path

# PART A: Load the existing Task 7 predictions

baseline = pd.read_parquet(TASK7_FILE)

### PART B: Inspect the train and validation splits

In [61]:
print("\nRows by split:")
print(baseline["split"].value_counts())

validation = baseline[
    baseline["split"] == "validation"
].copy()

print("\nValidation rows:", len(validation))

print(
    "Unique validation chunks:",
    validation["chunk_id"].nunique()
)

print(
    "Unique categories:",
    validation["category_id"].nunique()
)




Rows by split:
split
train         108410
validation     28730
Name: count, dtype: int64

Validation rows: 28730
Unique validation chunks: 2873
Unique categories: 10


### PART C: Check masks and predictions
Task 7 uses mask=1 for eligible pairs.
Masked pairs have mask=0 and must be excluded.

In [62]:
print("\nValidation mask counts:")
print(validation["mask"].value_counts(dropna=False))

print("\nPrediction columns:")

for column in [
    "lr_prediction",
    "keyword_prediction"
]:
    print(f"\n{column}:")
    print(validation[column].value_counts(dropna=False))


Validation mask counts:
mask
1    28690
0       40
Name: count, dtype: int64

Prediction columns:

lr_prediction:
lr_prediction
0    28061
1      669
Name: count, dtype: int64

keyword_prediction:
keyword_prediction
0    28486
1      244
Name: count, dtype: int64


### PART D: Check for duplicate pairs and missing values
We need one row per chunk and category.
Duplicates could cause the same example to be counted twice.

In [63]:
print("\nDuplicate chunk-category pairs:")

print(
    validation.duplicated(
        ["chunk_id", "category_id"]
    ).sum()
)

print("\nMissing values:")
print(validation.isna().sum())



Duplicate chunk-category pairs:
0

Missing values:
split                 0
chunk_id              0
category_id           0
label                 0
mask                  0
lr_score              0
lr_prediction         0
keyword_score         0
keyword_prediction    0
dtype: int64


### PART E: Inspect Task 8's frozen thresholds

In [64]:
thresholds = pd.read_csv(TASK8_THRESHOLDS)

print("\nTASK 8 FROZEN THRESHOLDS")
print("Shape:", thresholds.shape)
print("Columns:", thresholds.columns.tolist())

display(thresholds.head(10))

if "seed" in thresholds.columns:
    print("\nThreshold records by seed:")
    print(thresholds["seed"].value_counts())


TASK 8 FROZEN THRESHOLDS
Shape: (30, 6)
Columns: ['seed', 'category', 'threshold', 'precision', 'recall', 'f1']


,seed,category,threshold,precision,recall,f1
0,42,anti_assignment,0.90,0.901235,0.858824,0.879518
1,123,anti_assignment,0.75,0.891566,0.870588,0.880952
2,2026,anti_assignment,0.90,0.892857,0.882353,0.887574
3,42,audit_rights,0.95,0.909091,0.821918,0.863309
4,123,audit_rights,0.75,0.825000,0.904110,0.862745
5,2026,audit_rights,0.90,0.828947,0.863014,0.845638
6,42,cap_on_liability,0.75,0.840000,0.926471,0.881119
7,123,cap_on_liability,0.85,0.910448,0.897059,0.903704
8,2026,cap_on_liability,0.95,0.901639,0.808824,0.852713
9,42,exclusivity,0.95,0.781250,0.609756,0.684932



Threshold records by seed:
seed
42      10
123     10
2026    10
Name: count, dtype: int64


### PART F: Search the project for weighted score files
The files might be in another folder or a downloaded ZIP.
We search, but do not move or modify anything.

In [65]:
print("\nSEARCHING FOR WEIGHTED SCORE FILES...")

found_files = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if "weighted_legal-bert_seed" in name:
        found_files.append(path)

    elif name == "october_task4_outputs.zip":
        found_files.append(path)


if found_files:
    for path in found_files:
        print(path.relative_to(PROJECT_ROOT))
else:
    print("No matching weighted score files found locally.")

print(
    "\nOriginal weighted score files were not found in the searched locations. The team later supplied JSON prediction exports, which will be inspected in Step 3."
)


SEARCHING FOR WEIGHTED SCORE FILES...
No matching weighted score files found locally.

Original weighted score files were not found in the searched locations. The team later supplied JSON prediction exports, which will be inspected in Step 3.


# Step 3: Load and Inspect the Supplied Transformer Predicitons

The Task 7 baseline predictions are available locally. However, the original
weighted Legal-BERT score files were not located in the project directory.

Our team subsequently provided two JSON files containing previously generated chunk-level predictions:
- train_chunk_scores.json
- val_chunk_scores.json
We will use these existing files to inspect the transformer predictions and
prepare them for comparison against the Task 7 baselines.

We will not retrain the model or generate new predictions.

### Part A: Locate the supplied JSON files
Before loading the files, we need to tell Python where they are stored.

We use PROJECT_ROOT as the starting point because it already identifies the
project's notebooks directory. We then add the existing october_task11_outputs
folder to that path.

In [66]:
#Part A: Locate the supplied JSON predicted files

TASK11_INPUT_DIR = PROJECT_ROOT / "october_task11_outputs"

# Define the location of the training and validation files.
TRAIN_JSON_FILE = TASK11_INPUT_DIR / "train_chunk_scores.json"
VAL_JSON_FILE = TASK11_INPUT_DIR / "val_chunk_scores.json"

# Verify that both files are available before loading them.
print("Training JSON file found:", TRAIN_JSON_FILE.exists())
print("Validation JSON file found:", VAL_JSON_FILE.exists())

Training JSON file found: True
Validation JSON file found: True


### Part B: Load and inspect the JSON contents

We are using the json.load() to read each file. Resulting data will be stored in two seperate variables:
- train_json: contains the training chunk predictions
- val_json: contains the validation chunk predictions

We will also inspect the model version, number of chunks, and structure of the first chunk to understand
how predictions are organized.


In [67]:
import json

with open(TRAIN_JSON_FILE, "r", encoding="utf-8") as file:
    train_json = json.load(file)

with open(VAL_JSON_FILE, "r", encoding="utf-8") as file:
    val_json = json.load(file)

### PART C: Inspect the model version
This helps us document which model version is identified inside the supplied prediction files.

In [68]:
print("Training model version:", train_json["model_version"])
print("Validation model version:", val_json["model_version"])

Training model version: cuad_model_v1
Validation model version: cuad_model_v1


### PART D: Count the chunks in each split

chunk_predictions contains one record per chunk.
We inspect the counts to verify the expected coverage.

In [69]:
train_chunks = train_json["chunk_predictions"]
validation_chunks = val_json["chunk_predictions"]

print("\nTraining chunks:", len(train_chunks))
print("Validation chunks:", len(validation_chunks))


Training chunks: 10841
Validation chunks: 2873


### PART E: Inspect one validation chunk
We look at one example rather than printing the entire JSON file, which contains a large amount of contract text.
This helps us understand the fields available for analysis.

In [70]:
first_chunk = validation_chunks[0]

print("\nFIRST VALIDATION CHUNK")
print("Chunk ID:", first_chunk["chunk_id"])
print("Document ID:", first_chunk["document_id"])
print("Contract ID:", first_chunk["contract_id"])
print("Context Group:", first_chunk["context_group_id"])

print(
    "\nNumber of category results:",
    len(first_chunk["category_results"])
)

print("\nAvailable category fields:")

print(
    first_chunk["category_results"][0].keys()
)


FIRST VALIDATION CHUNK
Chunk ID: contract_0013__chunk_0000
Document ID: document_0013
Contract ID: contract_0013
Context Group: context_dc0f3df3b2

Number of category results: 10

Available category fields:
dict_keys(['category_id', 'score', 'threshold', 'predicted'])


### PART F: Display the ten category predictions
Each chunk should have a score, threshold, and prediction for each of the ten selected categories.

In [71]:
category_preview = pd.DataFrame(
    first_chunk["category_results"]
)
display(category_preview)

print("\nStep 3 inspection completed.")


,category_id,score,threshold,predicted
0,governing_law,0.0075,0.65,False
1,renewal_term,0.0128,0.95,False
2,revenue_profit_sharing,0.0156,0.95,False
3,cap_on_liability,0.0115,0.75,False
4,uncapped_liability,0.0064,0.90,False
5,termination_for_convenience,0.0139,0.85,False
6,anti_assignment,0.0139,0.90,False
7,audit_rights,0.0274,0.95,False
8,license_grant,0.0188,0.90,False
9,exclusivity,0.0404,0.95,False



Step 3 inspection completed.


### Part G: Validate Transformer Predictions Against Task 7
Before combining the transformer predictions with the Task 7 baseline data, we need to verify that both sources contain the same validation chunks and category assignments.
This is important because our comparison must evaluate the models on identical examples. Comparing different chunks or categories could produce misleading results.

We need to compare predictions on the SAME chunks and the SAME categories.
If the data does not line up, the model comparison would not be reliable.

Define the ten expected categories
We use the same category IDs established in the project.
This also gives us a consistent reference for later joins.

In [72]:
CATEGORY_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity"
]

### Compare validation chunk IDs
We extract the chunk IDs from both sources.
Sets make it easy to identify missing or extra IDs.

In [73]:
json_chunk_ids = set(
    chunk["chunk_id"] for chunk in validation_chunks
)

task7_chunk_ids = set(
    validation["chunk_id"].astype(str)
)

print("JSON validation chunks:", len(json_chunk_ids))
print("Task 7 validation chunks:", len(task7_chunk_ids))

# Identify chunks that appear in one source but not the other.
missing_from_json = task7_chunk_ids - json_chunk_ids
extra_in_json = json_chunk_ids - task7_chunk_ids

print("\nChunks missing from JSON:", len(missing_from_json))
print("Unexpected JSON chunks:", len(extra_in_json))


JSON validation chunks: 2873
Task 7 validation chunks: 2873

Chunks missing from JSON: 0
Unexpected JSON chunks: 0


#### Check category coverage
Each chunk should have exactly one result for each of the ten expected categories.

In [74]:
category_errors = 0

for chunk in validation_chunks:

    chunk_categories = [
        result["category_id"]
        for result in chunk["category_results"]
    ]

    if (
        len(chunk_categories) != 10
        or set(chunk_categories) != set(CATEGORY_IDS)
    ):
        category_errors += 1

print("\nChunks with incorrect category coverage:", category_errors)


Chunks with incorrect category coverage: 0


### Check for duplicate chunk-category predictions
Each chunk should have one prediction per category.
A duplicate could cause an example to be counted twice.

In [75]:
prediction_pairs = []

for chunk in validation_chunks:

    for result in chunk["category_results"]:

        prediction_pairs.append(
            (chunk["chunk_id"], result["category_id"])
        )

duplicate_pairs = (
    len(prediction_pairs) - len(set(prediction_pairs))
)

print("Duplicate chunk-category predictions:", duplicate_pairs)

Duplicate chunk-category predictions: 0


### Summarize validation checks
We report the checks together so the result is easy to review before moving to the comparison.

In [76]:
print("\nVALIDATION ALIGNMENT SUMMARY")

print("Matching chunk IDs:", len(json_chunk_ids & task7_chunk_ids))
print("Chunks missing from JSON:", len(missing_from_json))
print("Unexpected JSON chunks:", len(extra_in_json))
print("Category coverage errors:", category_errors)
print("Duplicate prediction pairs:", duplicate_pairs)

if (
    not missing_from_json
    and not extra_in_json
    and category_errors == 0
    and duplicate_pairs == 0
):
    print("\nAll validation alignment checks passed.")
else:
    print("\nSome checks need investigation before comparison.")



VALIDATION ALIGNMENT SUMMARY
Matching chunk IDs: 2873
Chunks missing from JSON: 0
Unexpected JSON chunks: 0
Category coverage errors: 0
Duplicate prediction pairs: 0

All validation alignment checks passed.


### Part H: VERIFY JSON THRESHOLDS AGAINST TASK 8
The supplied JSON contains category-specific thresholds.
Task 8 saved separate thresholds for each model seed.
We want to check whether the JSON thresholds match Seed 42.

#### Get the Seed 42 thresholds from Task 8
We filter the threshold table because it contains results for three different
seeds.

In [77]:
seed42_thresholds = thresholds[
    thresholds["seed"] == 42
][["category", "threshold"]].copy()

### Get the thresholds from the supplied JSON
We use the first validation chunk as the source because each chunk is expected to use the same category thresholds.

In [78]:
json_thresholds = pd.DataFrame(
    first_chunk["category_results"]
)[["category_id", "threshold"]]

### Match the two tables by category
The category column has different names in each source, so we specify which
columns should be matched.

In [79]:
threshold_comparison = seed42_thresholds.merge(
    json_thresholds,
    left_on="category",
    right_on="category_id",
    how="outer"
)

### Compare the threshold values
True means the JSON threshold matches Task 8 Seed 42.

In [80]:
import numpy as np

# Calculate the numerical difference between the two threshold values.
# This helps us identify whether any differences are only due to
# floating-point precision.
threshold_comparison["difference"] = (
    threshold_comparison["threshold_x"]
    - threshold_comparison["threshold_y"]
)

# Use np.isclose() instead of exact equality.
# A tolerance of 1e-9 allows extremely small numerical differences
# while still identifying meaningful threshold differences.
threshold_comparison["matches"] = np.isclose(
    threshold_comparison["threshold_x"],
    threshold_comparison["threshold_y"],
    rtol=0,
    atol=1e-9
)

# Display the values and their differences for all ten categories.
display(
    threshold_comparison[
        [
            "category",
            "threshold_x",
            "threshold_y",
            "difference",
            "matches"
        ]
    ]
)

# Summarize the comparison.
print(
    "\nMatching thresholds:",
    threshold_comparison["matches"].sum()
)

print(
    "Total categories:",
    len(threshold_comparison)
)

print(
    "All thresholds match Seed 42:",
    threshold_comparison["matches"].all()
)

,category,threshold_x,threshold_y,difference,matches
0,anti_assignment,0.90,0.90,1.110223e-16,True
1,audit_rights,0.95,0.95,0.000000e+00,True
2,cap_on_liability,0.75,0.75,1.110223e-16,True
3,exclusivity,0.95,0.95,0.000000e+00,True
4,governing_law,0.65,0.65,1.110223e-16,True
5,license_grant,0.90,0.90,1.110223e-16,True
6,renewal_term,0.95,0.95,0.000000e+00,True
7,revenue_profit_sharing,0.95,0.95,0.000000e+00,True
8,termination_for_convenience,0.85,0.85,1.110223e-16,True
9,uncapped_liability,0.90,0.90,1.110223e-16,True



Matching thresholds: 10
Total categories: 10
All thresholds match Seed 42: True


Summarize the result

In [81]:
print(
    "Matching thresholds:",
    threshold_comparison["matches"].sum()
)

print(
    "Total categories:",
    len(threshold_comparison)
)

print(
    "All thresholds match Seed 42:",
    threshold_comparison["matches"].all()
)


Matching thresholds: 10
Total categories: 10
All thresholds match Seed 42: True


### Part I: Verify thresholds across all validation chunks

Part H confirmed that the thresholds in the first validation chunk match the
Task 8 Seed 42 thresholds. In this section, we extend that check to the entire
validation JSON to confirm that the same category-specific thresholds were
used consistently across all 2,873 chunks and 28,730 chunk-category predictions.

The code creates a reference lookup using the Task 8 Seed 42 thresholds,
compares each JSON threshold against its corresponding category threshold,
and reports the number of matches and mismatches. A small tolerance is used
to account for floating-point precision differences.

This check helps establish threshold consistency across the validation
predictions before comparing the transformer against the two baselines.

In [82]:
seed42_threshold_map = dict(
    zip(
        seed42_thresholds["category"],
        seed42_thresholds["threshold"]
    )
)

# Create an empty list to store the threshold comparison results.
threshold_check_rows = []

# Go through every validation chunk.
for chunk in validation_chunks:

    # Go through all ten category predictions in the current chunk.
    for result in chunk["category_results"]:

        category_id = result["category_id"]
        json_threshold = result["threshold"]

        # Retrieve the corresponding Task 8 Seed 42 threshold.
        reference_threshold = seed42_threshold_map.get(category_id)

        # Check whether the category exists in the Task 8 threshold table.
        if reference_threshold is None:
            matches = False
            difference = None

        else:
            # Calculate the difference between the JSON and Task 8 thresholds.
            difference = json_threshold - reference_threshold

            # Allow for extremely small floating-point differences.
            matches = np.isclose(
                json_threshold,
                reference_threshold,
                rtol=0,
                atol=1e-9
            )

        # Save the comparison for this chunk and category.
        threshold_check_rows.append({
            "chunk_id": chunk["chunk_id"],
            "category_id": category_id,
            "json_threshold": json_threshold,
            "task8_threshold": reference_threshold,
            "difference": difference,
            "matches": matches
        })

# Convert the comparison results into a DataFrame.
threshold_check = pd.DataFrame(threshold_check_rows)

# Identify any predictions whose thresholds do not match Task 8 Seed 42.
threshold_mismatches = threshold_check[
    ~threshold_check["matches"]
]

# Print the overall results.
print("Total threshold comparisons:", len(threshold_check))
print("Matching thresholds:", threshold_check["matches"].sum())
print("Threshold mismatches:", len(threshold_mismatches))

print(
    "All validation thresholds match Seed 42:",
    threshold_check["matches"].all()
)

# Display mismatches if any are found.
if len(threshold_mismatches) > 0:
    display(threshold_mismatches.head(20))
else:
    print("\nNo threshold mismatches were found.")

Total threshold comparisons: 28730
Matching thresholds: 28730
Threshold mismatches: 0
All validation thresholds match Seed 42: True

No threshold mismatches were found.


## STEP 4: BUILD THE COMBINED VALIDATION COMPARISON TABLE

### Part A: Convert Transformer Predictions into a DataFrame

The supplied validation JSON stores predictions in a nested structure, with
multiple category results inside each chunk.

Before comparing the transformer against the Task 7 baselines, we need to
convert these predictions into a tabular format.

This section creates one row per chunk-category pair and retains the
transformer score, threshold, prediction, and associated contract text.

Keeping the contract text will also allow us to inspect individual examples
later when identifying recurring prediction errors and failure themes.

The resulting table should contain 28,730 rows, representing 2,873 validation
chunks across 10 categories.

### BUILD THE TRANSFORMER VALIDATION PREDICTIONS TABLE

In [83]:
# Create an empty list to store the flattened prediction records.
transformer_rows = []

# Go through each validation chunk in the supplied JSON file.
for chunk in validation_chunks:

    # Each chunk contains one prediction record for each category.
    for result in chunk["category_results"]:

        # Store the chunk information and its category prediction together.
        # This creates one record for each chunk-category pair.
        transformer_rows.append({

            # Identify the chunk and its related contract information.
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "contract_id": chunk["contract_id"],
            "context_group_id": chunk["context_group_id"],

            # Keep the original text and character positions.
            # These fields will help us inspect errors later.
            "chunk_text": chunk["chunk_text"],
            "char_start": chunk["char_start"],
            "char_end": chunk["char_end"],

            # Store the category and its transformer results.
            "category_id": result["category_id"],
            "transformer_score": result["score"],
            "transformer_threshold": result["threshold"],

            # Convert the Boolean prediction (True/False) into 1/0.
            # This makes it easier to compare against the baseline predictions.
            "transformer_prediction": int(result["predicted"])
        })


# Convert the list of records into a pandas DataFrame.
transformer_predictions = pd.DataFrame(transformer_rows)

# Standardize the ID columns as strings.
# This will help prevent data type mismatches when we merge tables later.
transformer_predictions["chunk_id"] = (
    transformer_predictions["chunk_id"].astype(str)
)

transformer_predictions["category_id"] = (
    transformer_predictions["category_id"].astype(str)
)

# Display the resulting table's dimensions.
print("TRANSFORMER VALIDATION TABLE")
print("Rows:", len(transformer_predictions))
print("Columns:", len(transformer_predictions.columns))

# Confirm the number of unique chunks and categories.
print(
    "\nUnique chunks:",
    transformer_predictions["chunk_id"].nunique()
)

print(
    "Unique categories:",
    transformer_predictions["category_id"].nunique()
)

# Check whether any chunk-category pairs appear more than once.
print(
    "\nDuplicate chunk-category pairs:",
    transformer_predictions.duplicated(
        ["chunk_id", "category_id"]
    ).sum()
)

# Inspect how many positive and negative predictions the transformer made.
print("\nTransformer prediction counts:")
print(
    transformer_predictions["transformer_prediction"].value_counts()
    .sort_index()
)

# Display a small sample to inspect the table structure.
display(transformer_predictions.head())

TRANSFORMER VALIDATION TABLE
Rows: 28730
Columns: 11

Unique chunks: 2873
Unique categories: 10

Duplicate chunk-category pairs: 0

Transformer prediction counts:
transformer_prediction
0    28135
1      595
Name: count, dtype: int64


,chunk_id,document_id,contract_id,context_group_id,chunk_text,char_start,char_end,category_id,transformer_score,transformer_threshold,transformer_prediction
0,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,governing_law,0.0075,0.65,0
1,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,renewal_term,0.0128,0.95,0
2,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,revenue_profit_sharing,0.0156,0.95,0
3,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,cap_on_liability,0.0115,0.75,0
4,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,uncapped_liability,0.0064,0.90,0


### Part B: Merge Transformer Predictions with Task 7 Baselines

In this section, we combine the transformer prediction table with the
existing Task 7 validation predictions.

The two tables will be matched using chunk_id and category_id. These
identifiers allow us to connect predictions for the same chunk and
category across all three models.

We retain the actual labels, eligibility masks, baseline scores, and
baseline predictions so that each model can be evaluated against the
same ground-truth labels.

An outer merge is used so that missing or unmatched pairs can be
identified instead of silently excluded.

At this stage, we retain both eligible and masked pairs. We will
separate eligible pairs in a later section after confirming that the
tables are correctly aligned.

### MERGE TRANSFORMER AND BASELINE PREDICTIONS
Create a separate copy of the Task 7 validation data.
This preserves the original validation DataFrame.

In [84]:
baseline_validation = validation.copy()

# Standardize the matching columns as strings in both tables.
# Consistent data types help prevent merge errors.
baseline_validation["chunk_id"] = (
    baseline_validation["chunk_id"].astype(str)
)

baseline_validation["category_id"] = (
    baseline_validation["category_id"].astype(str)
)

# Select only the baseline columns needed for the comparison.
# We retain the actual labels, masks, scores, and predictions.
baseline_columns = baseline_validation[
    [
        "chunk_id",
        "category_id",
        "label",
        "mask",
        "lr_score",
        "lr_prediction",
        "keyword_score",
        "keyword_prediction"
    ]
].copy()

### Merge the two prediction tables using chunk_id and category_id.
An outer merge allows us to identify pairs that appear in only one table.
validate="one_to_one" checks that neither table has duplicate matching pairs.

In [85]:
comparison = transformer_predictions.merge(
    baseline_columns,
    on=["chunk_id", "category_id"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

# Display the dimensions of the combined comparison table.
print("COMBINED VALIDATION COMPARISON TABLE")
print("Rows:", len(comparison))
print("Columns:", len(comparison.columns))

# Check whether every pair appears in both tables.
print("\nMerge alignment results:")
print(comparison["_merge"].value_counts())

# Count any unmatched pairs.
print(
    "\nTransformer-only pairs:",
    (comparison["_merge"] == "left_only").sum()
)

print(
    "Baseline-only pairs:",
    (comparison["_merge"] == "right_only").sum()
)

print(
    "Pairs found in both tables:",
    (comparison["_merge"] == "both").sum()
)

# Display a sample of the combined table.
display(comparison.head())

COMBINED VALIDATION COMPARISON TABLE
Rows: 28730
Columns: 18

Merge alignment results:
_merge
both          28730
left_only         0
right_only        0
Name: count, dtype: int64

Transformer-only pairs: 0
Baseline-only pairs: 0
Pairs found in both tables: 28730


,chunk_id,document_id,contract_id,context_group_id,chunk_text,char_start,char_end,category_id,transformer_score,transformer_threshold,transformer_prediction,label,mask,lr_score,lr_prediction,keyword_score,keyword_prediction,_merge
0,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,anti_assignment,0.0139,0.90,0,0,1,0.002482,0,0,0,both
1,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,audit_rights,0.0274,0.95,0,0,1,0.015619,0,1,1,both
2,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,cap_on_liability,0.0115,0.75,0,0,1,0.001239,0,0,0,both
3,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,exclusivity,0.0404,0.95,0,0,1,0.001548,0,0,0,both
4,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,governing_law,0.0075,0.65,0,0,1,0.008757,0,0,0,both


### Part C: Validate the Combined Table and Filter Eligible Pairs

Before evaluating the models, we need to confirm that the combined
comparison table contains complete and consistent data.

This section performs several checks:

1. Confirm that all three models have predictions for the same
   chunk-category pairs.
2. Check for duplicate chunk-category pairs.
3. Check for missing values in the required evaluation columns.
4. Confirm that labels, predictions, and masks contain valid values.
5. Separate eligible pairs from masked pairs.

Task 7 uses mask = 1 to identify eligible chunk-category pairs and
mask = 0 for excluded pairs.

The 40 masked pairs will be excluded from the model comparison.
The resulting eligible comparison dataset should contain 28,690 rows.

We retain the original combined table so that the masked records
remain available for reference and documentation.

### VALIDATE THE COMBINED TABLE AND FILTER ELIGIBLE PAIRS
Define the columns needed for the model comparison.
These fields must be present for us to evaluate predictions correctly.

In [86]:
required_columns = [
    "chunk_id",
    "category_id",
    "label",
    "mask",
    "transformer_prediction",
    "lr_prediction",
    "keyword_prediction"
]

# Check whether all required columns exist in the combined table.
missing_columns = [
    column for column in required_columns
    if column not in comparison.columns
]

print("MISSING REQUIRED COLUMNS:")
print(missing_columns)

# Check for duplicate chunk-category pairs.
# Each pair should appear only once in the combined table.
duplicate_pairs = comparison.duplicated(
    ["chunk_id", "category_id"]
).sum()

print("\nDuplicate chunk-category pairs:", duplicate_pairs)

# Check for missing values in the required evaluation columns.
# Missing labels or predictions could affect the comparison results.
print("\nMissing values in required columns:")

print(
    comparison[required_columns]
    .isna()
    .sum()
)

# Check that the labels, masks, and predictions contain only 0 and 1.
# These columns should represent binary values.
binary_columns = [
    "label",
    "mask",
    "transformer_prediction",
    "lr_prediction",
    "keyword_prediction"
]

print("\nUNIQUE VALUES IN BINARY COLUMNS:")

for column in binary_columns:
    print(f"{column}: {sorted(comparison[column].dropna().unique())}")

# Confirm that every row was matched during the merge.
# All rows should have the merge indicator value "both".
print("\nMerge alignment check:")
print(comparison["_merge"].value_counts())

# Count eligible and masked pairs before filtering.
print("\nEligibility mask counts:")
print(comparison["mask"].value_counts().sort_index())

# Create a separate dataset containing only eligible pairs.
# The original comparison table remains unchanged.
eligible_comparison = comparison[
    comparison["mask"] == 1
].copy()

# Count the number of excluded masked pairs.
excluded_pairs = comparison[
    comparison["mask"] == 0
]

# Display the final dataset dimensions and filtering results.
print("\nELIGIBLE COMPARISON DATASET")
print("Total combined pairs:", len(comparison))
print("Eligible pairs:", len(eligible_comparison))
print("Excluded masked pairs:", len(excluded_pairs))

# Confirm that the eligible dataset contains no duplicate pairs.
print(
    "Duplicate eligible pairs:",
    eligible_comparison.duplicated(
        ["chunk_id", "category_id"]
    ).sum()
)

# Check for missing values in the required evaluation columns
# after filtering to eligible pairs.
print(
    "\nMissing values in eligible evaluation columns:",
    eligible_comparison[required_columns].isna().sum().sum()
)

# Display a sample of the eligible comparison dataset.
display(eligible_comparison.head())

MISSING REQUIRED COLUMNS:
[]

Duplicate chunk-category pairs: 0

Missing values in required columns:
chunk_id                  0
category_id               0
label                     0
mask                      0
transformer_prediction    0
lr_prediction             0
keyword_prediction        0
dtype: int64

UNIQUE VALUES IN BINARY COLUMNS:
label: [np.int64(0), np.int64(1)]
mask: [np.int64(0), np.int64(1)]
transformer_prediction: [np.int64(0), np.int64(1)]
lr_prediction: [np.int64(0), np.int64(1)]
keyword_prediction: [np.int64(0), np.int64(1)]

Merge alignment check:
_merge
both          28730
left_only         0
right_only        0
Name: count, dtype: int64

Eligibility mask counts:
mask
0       40
1    28690
Name: count, dtype: int64

ELIGIBLE COMPARISON DATASET
Total combined pairs: 28730
Eligible pairs: 28690
Excluded masked pairs: 40
Duplicate eligible pairs: 0

Missing values in eligible evaluation columns: 0


,chunk_id,document_id,contract_id,context_group_id,chunk_text,char_start,char_end,category_id,transformer_score,transformer_threshold,transformer_prediction,label,mask,lr_score,lr_prediction,keyword_score,keyword_prediction,_merge
0,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,anti_assignment,0.0139,0.90,0,0,1,0.002482,0,0,0,both
1,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,audit_rights,0.0274,0.95,0,0,1,0.015619,0,1,1,both
2,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,cap_on_liability,0.0115,0.75,0,0,1,0.001239,0,0,0,both
3,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,exclusivity,0.0404,0.95,0,0,1,0.001548,0,0,0,both
4,contract_0013__chunk_0000,document_0013,contract_0013,context_dc0f3df3b2,DISTRIBUTION AND SERVICES AGREEMENT January 18...,0,2676,governing_law,0.0075,0.65,0,0,1,0.008757,0,0,0,both


## STEP 5: COMPARE MODEL CORRECTNESS AND IDENTIFY ERRORS

### Part A: Classify Prediction Outcomes

Now that the three prediction tables have been aligned, we can compare
their predictions against the ground-truth labels.

We will evaluate the transformer separately against each baseline:

1. Logistic regression.
2. Keyword baseline.

For each comparison, every eligible chunk-category pair will be assigned
to one of four outcome groups:

- **Newly corrected:** The baseline prediction was incorrect, but the
  transformer prediction was correct.
- **New error:** The baseline prediction was correct, but the transformer
  prediction was incorrect.
- **Persistent error:** Both the baseline and transformer predictions
  were incorrect.
- **Both correct:** Both models made the correct prediction.

We use the actual label column as the reference for determining whether
a prediction is correct.

The two baseline comparisons remain separate because each baseline may
have different strengths and weaknesses.

The analysis uses only the 28,690 eligible pairs. No masked pairs are
included.

In [87]:
error_comparison = eligible_comparison.copy()

# First, determine whether each model's prediction matches the actual label.
# A result of True means the prediction was correct.
error_comparison["transformer_correct"] = (
    error_comparison["transformer_prediction"]
    == error_comparison["label"]
)

error_comparison["lr_correct"] = (
    error_comparison["lr_prediction"]
    == error_comparison["label"]
)

error_comparison["keyword_correct"] = (
    error_comparison["keyword_prediction"]
    == error_comparison["label"]
)

### TRANSFORMER VS. LOGISTIC REGRESSION

Newly corrected: Logistic regression was wrong, but the transformer was correct.

In [88]:
error_comparison["lr_newly_corrected"] = (
    (~error_comparison["lr_correct"])
    & error_comparison["transformer_correct"]
)

# New error:
# Logistic regression was correct, but the transformer was wrong.
error_comparison["lr_new_error"] = (
    error_comparison["lr_correct"]
    & (~error_comparison["transformer_correct"])
)

# Persistent error:
# Both models made an incorrect prediction.
error_comparison["lr_persistent_error"] = (
    (~error_comparison["lr_correct"])
    & (~error_comparison["transformer_correct"])
)

# Both correct:
# Both models made the correct prediction.
error_comparison["lr_both_correct"] = (
    error_comparison["lr_correct"]
    & error_comparison["transformer_correct"]
)

#### TRANSFORMER VS. KEYWORD BASELINE

In [89]:
# Newly corrected:
# The keyword baseline was wrong, but the transformer was correct
error_comparison["keyword_newly_corrected"] = (
    (~error_comparison["keyword_correct"])
    & error_comparison["transformer_correct"]
)

# New error:
# The keyword baseline was correct, but the transformer was wrong.
error_comparison["keyword_new_error"] = (
    error_comparison["keyword_correct"]
    & (~error_comparison["transformer_correct"])
)

# Persistent error:
# Both models made an incorrect prediction.
error_comparison["keyword_persistent_error"] = (
    (~error_comparison["keyword_correct"])
    & (~error_comparison["transformer_correct"])
)

# Both correct:
# Both models made the correct prediction.
error_comparison["keyword_both_correct"] = (
    error_comparison["keyword_correct"]
    & error_comparison["transformer_correct"]
)

### SUMMARIZE THE OUTCOMES
Count the four outcome groups for each baseline comparison.

In [90]:
outcome_summary = pd.DataFrame({
    "Outcome": [
        "Newly corrected",
        "New error",
        "Persistent error",
        "Both correct"
    ],

    "Transformer vs. Logistic Regression": [
        error_comparison["lr_newly_corrected"].sum(),
        error_comparison["lr_new_error"].sum(),
        error_comparison["lr_persistent_error"].sum(),
        error_comparison["lr_both_correct"].sum()
    ],

    "Transformer vs. Keyword Baseline": [
        error_comparison["keyword_newly_corrected"].sum(),
        error_comparison["keyword_new_error"].sum(),
        error_comparison["keyword_persistent_error"].sum(),
        error_comparison["keyword_both_correct"].sum()
    ]
})

# Display the comparison results.
print("MODEL COMPARISON OUTCOMES")
display(outcome_summary)

# Confirm that every eligible pair belongs to exactly one outcome
# in each of the two baseline comparisons.
lr_outcome_total = (
    error_comparison["lr_newly_corrected"].sum()
    + error_comparison["lr_new_error"].sum()
    + error_comparison["lr_persistent_error"].sum()
    + error_comparison["lr_both_correct"].sum()
)

keyword_outcome_total = (
    error_comparison["keyword_newly_corrected"].sum()
    + error_comparison["keyword_new_error"].sum()
    + error_comparison["keyword_persistent_error"].sum()
    + error_comparison["keyword_both_correct"].sum()
)

print("\nOUTCOME TOTAL CHECK")
print("Eligible pairs:", len(error_comparison))
print("Logistic regression comparison total:", lr_outcome_total)
print("Keyword comparison total:", keyword_outcome_total)

print(
    "\nAll logistic regression outcomes accounted for:",
    lr_outcome_total == len(error_comparison)
)

print(
    "All keyword outcomes accounted for:",
    keyword_outcome_total == len(error_comparison)
)

MODEL COMPARISON OUTCOMES


,Outcome,Transformer vs. Logistic Regression,Transformer vs. Keyword Baseline
0,Newly corrected,396,446
1,New error,91,106
2,Persistent error,114,99
3,Both correct,28089,28039



OUTCOME TOTAL CHECK
Eligible pairs: 28690
Logistic regression comparison total: 28690
Keyword comparison total: 28690

All logistic regression outcomes accounted for: True
All keyword outcomes accounted for: True


### Part B: Compare Prediction Outcomes by Category

The overall results show how the transformer performed against each
baseline across all eligible validation pairs.

However, overall counts do not show whether improvements and errors
are concentrated in particular contract categories.

In this section, we break down the four prediction outcomes by category
for both baseline comparisons:

1. Transformer versus logistic regression.
2. Transformer versus keyword baseline.

For each category, we calculate:

- Newly corrected errors.
- New errors introduced by the transformer.
- Persistent errors.
- Pairs where both models were correct.
- Total eligible pairs.

We also calculate the percentage of eligible pairs represented by
newly corrected errors and new errors within each category.

This breakdown will help identify categories where the transformer
shows more improvement, introduces more errors, or continues to
struggle.

All results are based on the eligible validation pairs only.

### COMPARE MODEL OUTCOMES BY CATEGORY

TRANSFORMER VS. LOGISTIC REGRESSION

In [91]:
# Group the eligible comparison data by contract category.
# Count how many pairs belong to each outcome group.
lr_category_summary = error_comparison.groupby("category_id").agg(

    # Count the total number of eligible pairs in each category.
    eligible_pairs=("label", "size"),

    # Count pairs where the transformer corrected a logistic regression error.
    newly_corrected=("lr_newly_corrected", "sum"),

    # Count pairs where the transformer introduced a new error.
    new_errors=("lr_new_error", "sum"),

    # Count pairs where both models made an incorrect prediction.
    persistent_errors=("lr_persistent_error", "sum"),

    # Count pairs where both models made a correct prediction.
    both_correct=("lr_both_correct", "sum")

).reset_index()

# Calculate the percentage of eligible pairs that were newly corrected.
lr_category_summary["newly_corrected_pct"] = (
    lr_category_summary["newly_corrected"]
    / lr_category_summary["eligible_pairs"]
    * 100
)

# Calculate the percentage of eligible pairs where the transformer
# introduced a new error.
lr_category_summary["new_errors_pct"] = (
    lr_category_summary["new_errors"]
    / lr_category_summary["eligible_pairs"]
    * 100
)

# Calculate the net difference between corrected errors and new errors.
# A positive value means more errors were corrected than introduced.
lr_category_summary["net_error_reduction"] = (
    lr_category_summary["newly_corrected"]
    - lr_category_summary["new_errors"]
)

# Round percentages to two decimal places for readability.
lr_category_summary["newly_corrected_pct"] = (
    lr_category_summary["newly_corrected_pct"].round(2)
)

lr_category_summary["new_errors_pct"] = (
    lr_category_summary["new_errors_pct"].round(2)
)

# Sort categories by the number of newly corrected errors.
lr_category_summary = lr_category_summary.sort_values(
    "newly_corrected",
    ascending=False
)

print("TRANSFORMER VS. LOGISTIC REGRESSION BY CATEGORY")
display(lr_category_summary)


TRANSFORMER VS. LOGISTIC REGRESSION BY CATEGORY


,category_id,eligible_pairs,newly_corrected,new_errors,persistent_errors,both_correct,newly_corrected_pct,new_errors_pct,net_error_reduction
2,cap_on_liability,2869,70,5,12,2782,2.44,0.17,65
4,governing_law,2873,59,2,4,2808,2.05,0.07,57
0,anti_assignment,2872,57,7,13,2795,1.98,0.24,50
5,license_grant,2860,56,16,16,2772,1.96,0.56,40
8,termination_for_convenience,2872,36,10,10,2816,1.25,0.35,26
7,revenue_profit_sharing,2867,34,8,12,2813,1.19,0.28,26
1,audit_rights,2870,30,7,12,2821,1.05,0.24,23
3,exclusivity,2866,26,9,14,2817,0.91,0.31,17
6,renewal_term,2869,18,7,8,2836,0.63,0.24,11
9,uncapped_liability,2872,10,20,13,2829,0.35,0.70,-10


### TRANSFORMER VS. KEYWORD BASELINE


In [92]:
# Group the eligible comparison data by contract category.
# Count the four outcome groups for the keyword comparison.
keyword_category_summary = error_comparison.groupby("category_id").agg(

    # Count the total number of eligible pairs in each category.
    eligible_pairs=("label", "size"),

    # Count pairs where the transformer corrected a keyword baseline error.
    newly_corrected=("keyword_newly_corrected", "sum"),

    # Count pairs where the transformer introduced a new error.
    new_errors=("keyword_new_error", "sum"),

    # Count pairs where both models made an incorrect prediction.
    persistent_errors=("keyword_persistent_error", "sum"),

    # Count pairs where both models made a correct prediction.
    both_correct=("keyword_both_correct", "sum")

).reset_index()

# Calculate the percentage of eligible pairs that were newly corrected.
keyword_category_summary["newly_corrected_pct"] = (
    keyword_category_summary["newly_corrected"]
    / keyword_category_summary["eligible_pairs"]
    * 100
)

# Calculate the percentage of eligible pairs where the transformer
# introduced a new error.
keyword_category_summary["new_errors_pct"] = (
    keyword_category_summary["new_errors"]
    / keyword_category_summary["eligible_pairs"]
    * 100
)

# Calculate the net difference between corrected errors and new errors.
keyword_category_summary["net_error_reduction"] = (
    keyword_category_summary["newly_corrected"]
    - keyword_category_summary["new_errors"]
)

# Round percentages to two decimal places for readability.
keyword_category_summary["newly_corrected_pct"] = (
    keyword_category_summary["newly_corrected_pct"].round(2)
)

keyword_category_summary["new_errors_pct"] = (
    keyword_category_summary["new_errors_pct"].round(2)
)

# Sort categories by the number of newly corrected errors.
keyword_category_summary = keyword_category_summary.sort_values(
    "newly_corrected",
    ascending=False
)

print("\nTRANSFORMER VS. KEYWORD BASELINE BY CATEGORY")
display(keyword_category_summary)



TRANSFORMER VS. KEYWORD BASELINE BY CATEGORY


,category_id,eligible_pairs,newly_corrected,new_errors,persistent_errors,both_correct,newly_corrected_pct,new_errors_pct,net_error_reduction
0,anti_assignment,2872,74,8,12,2778,2.58,0.28,66
2,cap_on_liability,2869,68,12,5,2784,2.37,0.42,56
8,termination_for_convenience,2872,61,13,7,2791,2.12,0.45,48
1,audit_rights,2870,57,5,14,2794,1.99,0.17,52
4,governing_law,2873,57,4,2,2810,1.98,0.14,53
5,license_grant,2860,42,14,18,2786,1.47,0.49,28
7,revenue_profit_sharing,2867,24,6,14,2823,0.84,0.21,18
3,exclusivity,2866,23,7,16,2820,0.80,0.24,16
6,renewal_term,2869,20,7,8,2834,0.70,0.24,13
9,uncapped_liability,2872,20,30,3,2819,0.70,1.04,-10


## VALIDATE THE CATEGORY SUMMARIES

Confirm that the four outcome groups add up to the eligible
pair count for every category in the logistic regression comparison.

In [93]:
lr_category_summary["outcomes_accounted_for"] = (
    lr_category_summary["newly_corrected"]
    + lr_category_summary["new_errors"]
    + lr_category_summary["persistent_errors"]
    + lr_category_summary["both_correct"]
    == lr_category_summary["eligible_pairs"]
)

# Perform the same check for the keyword comparison.
keyword_category_summary["outcomes_accounted_for"] = (
    keyword_category_summary["newly_corrected"]
    + keyword_category_summary["new_errors"]
    + keyword_category_summary["persistent_errors"]
    + keyword_category_summary["both_correct"]
    == keyword_category_summary["eligible_pairs"]
)

# Print the validation results.
print("\nCATEGORY SUMMARY VALIDATION")

print(
    "Logistic regression categories accounted for:",
    lr_category_summary["outcomes_accounted_for"].all()
)

print(
    "Keyword baseline categories accounted for:",
    keyword_category_summary["outcomes_accounted_for"].all()
)

print(
    "\nNumber of logistic regression categories:",
    len(lr_category_summary)
)

print(
    "Number of keyword baseline categories:",
    len(keyword_category_summary)
)


CATEGORY SUMMARY VALIDATION
Logistic regression categories accounted for: True
Keyword baseline categories accounted for: True

Number of logistic regression categories: 10
Number of keyword baseline categories: 10


### Part C: Identify Transformer False Positives and False Negatives

The previous sections identified how often the transformer corrected
baseline errors or introduced new ones.

In this section, we isolate the transformer's incorrect predictions
and separate them into two groups:

- **False positives (FP):** The transformer predicted that a category
  was present, but the actual label was 0.
- **False negatives (FN):** The transformer predicted that a category
  was absent, but the actual label was 1.

We will count these errors by category and retain the associated
contract text, chunk identifiers, transformer scores, and baseline
predictions.

These records will provide examples for the later manual review of
failure themes.

The analysis uses only eligible validation pairs.

#### IDENTIFY TRANSFORMER FALSE POSITIVES AND FALSE NEGATIVES

In [94]:
# A false positive occurs when the transformer predicts 1
# but the actual ground-truth label is 0.
transformer_false_positives = error_comparison[
    (error_comparison["transformer_prediction"] == 1)
    & (error_comparison["label"] == 0)
].copy()

# A false negative occurs when the transformer predicts 0
# but the actual ground-truth label is 1.
transformer_false_negatives = error_comparison[
    (error_comparison["transformer_prediction"] == 0)
    & (error_comparison["label"] == 1)
].copy()

#### COUNT FALSE POSITIVES AND FALSE NEGATIVES

In [95]:
print("TRANSFORMER ERROR COUNTS")

print(
    "Total false positives:",
    len(transformer_false_positives)
)

print(
    "Total false negatives:",
    len(transformer_false_negatives)
)

print(
    "Total transformer errors:",
    len(transformer_false_positives)
    + len(transformer_false_negatives)
)


TRANSFORMER ERROR COUNTS
Total false positives: 110
Total false negatives: 95
Total transformer errors: 205


#### SUMMARIZE ERRORS BY CATEGORY

In [96]:
# Count false positives in each category.
false_positive_summary = (
    transformer_false_positives
    .groupby("category_id")
    .size()
    .reset_index(name="false_positives")
)

# Count false negatives in each category.
false_negative_summary = (
    transformer_false_negatives
    .groupby("category_id")
    .size()
    .reset_index(name="false_negatives")
)

# Combine both error summaries so that every category is represented.
transformer_error_summary = false_positive_summary.merge(
    false_negative_summary,
    on="category_id",
    how="outer"
)

# Replace missing counts with zero.
# A category may have false positives but no false negatives, or vice versa.
transformer_error_summary[
    ["false_positives", "false_negatives"]
] = transformer_error_summary[
    ["false_positives", "false_negatives"]
].fillna(0).astype(int)

# Calculate the total transformer errors for each category.
transformer_error_summary["total_errors"] = (
    transformer_error_summary["false_positives"]
    + transformer_error_summary["false_negatives"]
)

# Sort categories by their total number of transformer errors.
transformer_error_summary = transformer_error_summary.sort_values(
    "total_errors",
    ascending=False
)

print("\nTRANSFORMER FALSE POSITIVES AND FALSE NEGATIVES BY CATEGORY")
display(transformer_error_summary)



TRANSFORMER FALSE POSITIVES AND FALSE NEGATIVES BY CATEGORY


,category_id,false_positives,false_negatives,total_errors
9,uncapped_liability,29,4,33
5,license_grant,14,18,32
3,exclusivity,7,16,23
0,anti_assignment,8,12,20
8,termination_for_convenience,17,3,20
7,revenue_profit_sharing,6,14,20
1,audit_rights,6,13,19
2,cap_on_liability,12,5,17
6,renewal_term,7,8,15
4,governing_law,4,2,6


#### VALIDATE THE ERROR COUNTS

In [97]:
# Confirm that the false-positive and false-negative counts
# account for all incorrect transformer predictions.
print("\nERROR COUNT VALIDATION")

print(
    "False-positive records match:",
    len(transformer_false_positives)
    == (error_comparison["transformer_prediction"].eq(1)
        & error_comparison["label"].eq(0)).sum()
)

print(
    "False-negative records match:",
    len(transformer_false_negatives)
    == (error_comparison["transformer_prediction"].eq(0)
        & error_comparison["label"].eq(1)).sum()
)

print(
    "Category error totals reconcile:",
    transformer_error_summary["total_errors"].sum()
    == len(transformer_false_positives)
    + len(transformer_false_negatives)
)


ERROR COUNT VALIDATION
False-positive records match: True
False-negative records match: True
Category error totals reconcile: True


### Part D: Select Transformer Error Examples for Manual Review

The previous section identified the transformer's false positives and
false negatives across all ten categories.

For an initial manual review, we will focus on the three categories
with the highest total number of transformer errors:

- Uncapped liability
- License grant
- Exclusivity

We will select up to three examples from each error type within
each category.

The selection method considers the transformer's score relative
to its category-specific threshold:

- For false positives, we select examples with scores furthest
  above the threshold.
- For false negatives, we select examples with scores closest
  to the threshold from below.

This allows us to inspect both high-scoring incorrect predictions
and borderline missed predictions.

Each selected record retains its contract text, category, ground-truth
label, transformer score and threshold, and both baseline predictions.

The complete false-positive and false-negative datasets remain
available for additional review. The selected examples are an
initial inspection sample, not a statistically representative sample.

No failure themes will be assigned until the actual contract text
has been reviewed.

In [98]:
# SELECT TRANSFORMER ERROR EXAMPLES FOR MANUAL REVIEW
# SELECT THE TOP THREE ERROR CATEGORIES

# Use the existing error summary to identify the three categories
# with the highest total number of transformer errors.
top_error_categories = (
    transformer_error_summary
    .head(3)["category_id"]
    .tolist()
)

print("CATEGORIES SELECTED FOR MANUAL REVIEW:")
for category in top_error_categories:
    print(category)

# SELECT FALSE POSITIVE EXAMPLES

# Create an empty list to store the selected false-positive examples.
fp_review_rows = []

# Review each of the three selected categories.
for category in top_error_categories:

    # Filter the complete false-positive table to the current category.
    category_fp = transformer_false_positives[
        transformer_false_positives["category_id"] == category
    ].copy()

    # Calculate how far the transformer's score is above its threshold.
    # A larger positive difference means a higher-scoring false positive.
    category_fp["threshold_distance"] = (
        category_fp["transformer_score"]
        - category_fp["transformer_threshold"]
    )

    # Select up to three false positives with the largest score margins.
    category_fp = category_fp.sort_values(
        "threshold_distance",
        ascending=False
    ).head(3)

    # Add the selected examples to the review list.
    fp_review_rows.append(category_fp)


# Combine the selected false-positive examples into one DataFrame.
fp_review_sample = pd.concat(
    fp_review_rows,
    ignore_index=True
)

# Identify these records as false positives.
fp_review_sample["error_type"] = "False positive"

# SELECT FALSE NEGATIVE EXAMPLES

# Create an empty list to store the selected false-negative examples.
fn_review_rows = []

# Review each of the three selected categories.
for category in top_error_categories:

    # Filter the complete false-negative table to the current category.
    category_fn = transformer_false_negatives[
        transformer_false_negatives["category_id"] == category
    ].copy()

    # Calculate how far the transformer's score is below its threshold.
    # A smaller positive difference means the prediction was closer
    # to crossing the threshold.
    category_fn["threshold_distance"] = (
        category_fn["transformer_threshold"]
        - category_fn["transformer_score"]
    )

    # Select up to three false negatives closest to the threshold.
    category_fn = category_fn.sort_values(
        "threshold_distance",
        ascending=True
    ).head(3)

    # Add the selected examples to the review list.
    fn_review_rows.append(category_fn)


# Combine the selected false-negative examples into one DataFrame.
fn_review_sample = pd.concat(
    fn_review_rows,
    ignore_index=True
)

# Identify these records as false negatives.
fn_review_sample["error_type"] = "False negative"

# COMBINE THE REVIEW EXAMPLES

# Combine the false-positive and false-negative samples.
# The complete source DataFrames remain unchanged.
transformer_error_review = pd.concat(
    [
        fp_review_sample,
        fn_review_sample
    ],
    ignore_index=True
)

# Arrange the examples by category and error type.
transformer_error_review = transformer_error_review.sort_values(
    ["category_id", "error_type"]
).reset_index(drop=True)

# SUMMARIZE THE SELECTED EXAMPLES

print("\nSELECTED ERROR EXAMPLES")
print("Total examples selected:", len(transformer_error_review))

# Count the number of selected examples for each category and error type.
review_sample_summary = (
    transformer_error_review
    .groupby(["category_id", "error_type"])
    .size()
    .unstack(fill_value=0)
)

display(review_sample_summary)

# DISPLAY THE CONTRACT TEXT AND PREDICTION DETAILS

# Keep the fields needed to understand each prediction.
review_columns = [
    "chunk_id",
    "contract_id",
    "category_id",
    "error_type",
    "label",
    "transformer_prediction",
    "transformer_score",
    "transformer_threshold",
    "threshold_distance",
    "lr_prediction",
    "keyword_prediction",
    "chunk_text"
]

print("\nCONTRACT EXAMPLES FOR MANUAL REVIEW")

# Display the full contract text for the selected examples.
# The option context allows long text to be displayed without truncation.
with pd.option_context("display.max_colwidth", None):
    display(transformer_error_review[review_columns])

CATEGORIES SELECTED FOR MANUAL REVIEW:
uncapped_liability
license_grant
exclusivity

SELECTED ERROR EXAMPLES
Total examples selected: 18


error_type,False negative,False positive
category_id,,
exclusivity,3,3
license_grant,3,3
uncapped_liability,3,3



CONTRACT EXAMPLES FOR MANUAL REVIEW


,chunk_id,contract_id,category_id,error_type,label,transformer_prediction,transformer_score,transformer_threshold,threshold_distance,lr_prediction,keyword_prediction,chunk_text
0,contract_0195__chunk_0003,contract_0195,exclusivity,False negative,1,0,0.9485,0.95,0.0015,1,0,"(H) TRANSPORTER. SRP is attempting to purchase or lease a 53 foot semi-trailer to be pulled by a Peterbilt class eight tractor which will function as the Race Car transporter (the ""Transporter""). In the event SRP is successful in procuring the Transporter, it will be lettered and pictured with Logan's racing graphics by SRP up to a maximum cost allowance of Two Thousand Five Hundred and No/100's ($2,500.00), with Logan's retaining the right to apply more extensive graphics at its own expense if Logan's so chooses. Any associate sponsor graphics appearing on the Transporter will be displayed in a subservient manner to clearly reflect Logan's as the primary sponsor of the Team.\n\n (I) AUTHORITY OF SRP. SRP hereby warrants that it has the authority to grant all rights to Baker under this Article 3.\n\n 4. ASSOCIATE AND SECONDARY SPONSORSHIPS. SRP shall have the right to obtain secondary and associate sponsors for the Race Car and Team, subject to the approval of Logan's, which approval shall not be unreasonably withheld. SRP may not obtain any associate or secondary sponsors whose products or concepts compete with Logan's. Any associate and/or secondary sponsor may promote its role as a sponsor by signage, logos or trademarks on the Race Car, provided that such associate and/or secondary sponsors signage, logos or trademarks used on the Race Car, when viewed collectively, shall not cover any area greater than 25% of that of Logan's signage, logos or trademarks, nor conflict with the graphic design, or in other locations or on clothing as may be permitted by the terms of the Agreement. Logan's shall have the exclusive use of the hood, the rear quarter panel area above the tire and the bottom of the deck lid (rear facing panel, TV panel) of the Race Car, as shown on Exhibit C to this Agreement. All other areas permitted by NASCAR for sponsor signage, logos or trademarks and the ""spoiler space"" are reserved for NASCAR, SRP and the associate and secondary sponsors. All associate and secondary sponsors' graphic layouts for the Race Car are subject to Logan's approval, which shall not be unreasonably withheld. Furthermore, SRP shall not permit any associate or secondary sponsor to suggest in any manner that its sponsorship role is as great as that of Logan's."
1,contract_0366__chunk_0013,contract_0366,exclusivity,False negative,1,0,0.9475,0.95,0.0025,0,0,"patent applications and other Intellectual Property Rights related to such inventions, if any, unless otherwise expressly set forth herein. During the Term, each Party hereby provides a worldwide, exclusive, royalty free, perpetual license of such Intellectual Property Rights for use by each licensee in its business in connection with the development and marketing and commercialization of the Product. For avoidance of doubt, no Party or any of its Affiliates shall have any rights to the Background IP of the other Party nor shall any Party have rights to any trademarks, service marks, trade names, business names or product names developed by the other Party. All decisions regarding the protection and exploitation of joint investments and other Intellectual Property Rights shall be determined by the JPT or otherwise in accordance with Section 2.1. 3.7 [reserved] CERTAIN PORTIONS OF THIS EXHIBIT HAVE BEEN OMITTED PURSUANT TO A REQUEST FOR CONFIDENTIAL TREATMENT UNDER RULE 24B-2 UNDER THE SECURITIES EXCHANGE ACT OF 1934. OMISSIONS ARE DESIGNATED [***]. A COMPLETE VERSION OF THIS EXHIBIT HAS BEEN FILED SEPARATELY WITH THE SECURITIES AND EXCHANGE COMMISSION. 7"
2,contract_0195__chunk_0002,contract_0195,exclusivity,False negative,1,0,0.7798,0.95,0.1702,0,0,"(E) UNIFORMS. SRP will provide race day uniforms for up to a 

### Part E: Prepare the Manual Error Review Table

The selected examples contain false positives and false negatives
from the three categories with the highest number of transformer errors.

Before assigning failure themes, we need to review the actual contract
language and determine whether each prediction represents a genuine
model error, a possible annotation issue, or an ambiguous example.

This section adds four fields for manual review:

- **Manual assessment:** The reviewer's assessment of the prediction.
- **Candidate failure theme:** A potential recurring pattern observed
  in the contract language.
- **Label needs verification:** Whether the ground-truth annotation
  should be reviewed.
- **Review notes:** Evidence supporting the assessment.

These fields will initially remain unassigned. They should only be
completed after examining each contract passage.

The original predictions, scores, thresholds, and labels will remain
unchanged.

In [99]:
# PREPARE THE MANUAL ERROR REVIEW TABLE

# Create a separate copy of the selected transformer error examples.
# This protects the original error review dataset.
manual_review = transformer_error_review.copy()

# Add a field for the reviewer's assessment of each prediction.
# Leave the assessment pending until the contract text is examined.
manual_review["manual_assessment"] = "Pending review"

# Add a field for a potential failure theme.
# We leave this blank so that themes are not assigned prematurely.
manual_review["candidate_failure_theme"] = ""

# Add a field to indicate whether the ground-truth label needs verification.
# The reviewer will determine this after examining the contract language.
manual_review["label_needs_verification"] = "Pending review"

# Add a field for notes explaining the reviewer's reasoning.
manual_review["review_notes"] = ""

# ORGANIZE THE REVIEW TABLE

# Select the fields needed for the manual review.
# The original model outputs and contract text are retained.
manual_review_columns = [
    "chunk_id",
    "contract_id",
    "category_id",
    "error_type",
    "label",
    "transformer_prediction",
    "transformer_score",
    "transformer_threshold",
    "lr_prediction",
    "keyword_prediction",
    "manual_assessment",
    "candidate_failure_theme",
    "label_needs_verification",
    "review_notes",
    "chunk_text"
]

# Create the review table using the selected columns.
manual_review = manual_review[manual_review_columns].copy()

# CHECK THE REVIEW TABLE

print("MANUAL ERROR REVIEW TABLE")

print("\nTotal examples:", len(manual_review))

print("\nExamples by error type:")
print(manual_review["error_type"].value_counts())

print("\nExamples by category and error type:")
print(
    manual_review.groupby(
        ["category_id", "error_type"]
    ).size()
)

# DISPLAY THE REVIEW TABLE

# Display the contract text without truncating the column.
# This allows us to inspect the actual language in each example.
with pd.option_context("display.max_colwidth", None):
    display(manual_review)

MANUAL ERROR REVIEW TABLE

Total examples: 18

Examples by error type:
error_type
False negative    9
False positive    9
Name: count, dtype: int64

Examples by category and error type:
category_id         error_type    
exclusivity         False negative    3
                    False positive    3
license_grant       False negative    3
                    False positive    3
uncapped_liability  False negative    3
                    False positive    3
dtype: int64


,chunk_id,contract_id,category_id,error_type,label,transformer_prediction,transformer_score,transformer_threshold,lr_prediction,keyword_prediction,manual_assessment,candidate_failure_theme,label_needs_verification,review_notes,chunk_text
0,contract_0195__chunk_0003,contract_0195,exclusivity,False negative,1,0,0.9485,0.95,1,0,Pending review,,Pending review,,"(H) TRANSPORTER. SRP is attempting to purchase or lease a 53 foot semi-trailer to be pulled by a Peterbilt class eight tractor which will function as the Race Car transporter (the ""Transporter""). In the event SRP is successful in procuring the Transporter, it will be lettered and pictured with Logan's racing graphics by SRP up to a maximum cost allowance of Two Thousand Five Hundred and No/100's ($2,500.00), with Logan's retaining the right to apply more extensive graphics at its own expense if Logan's so chooses. Any associate sponsor graphics appearing on the Transporter will be displayed in a subservient manner to clearly reflect Logan's as the primary sponsor of the Team.\n\n (I) AUTHORITY OF SRP. SRP hereby warrants that it has the authority to grant all rights to Baker under this Article 3.\n\n 4. ASSOCIATE AND SECONDARY SPONSORSHIPS. SRP shall have the right to obtain secondary and associate sponsors for the Race Car and Team, subject to the approval of Logan's, which approval shall not be unreasonably withheld. SRP may not obtain any associate or secondary sponsors whose products or concepts compete with Logan's. Any associate and/or secondary sponsor may promote its role as a sponsor by signage, logos or trademarks on the Race Car, provided that such associate and/or secondary sponsors signage, logos or trademarks used on the Race Car, when viewed collectively, shall not cover any area greater than 25% of that of Logan's signage, logos or trademarks, nor conflict with the graphic design, or in other locations or on clothing as may be permitted by the terms of the Agreement. Logan's shall have the exclusive use of the hood, the rear quarter panel area above the tire and the bottom of the deck lid (rear facing panel, TV panel) of the Race Car, as shown on Exhibit C to this Agreement. All other areas permitted by NASCAR for sponsor signage, logos or trademarks and the ""spoiler space"" are reserved for NASCAR, SRP and the associate and secondary sponsors. All associate and secondary sponsors' graphic layouts for the Race Car are subject to Logan's approval, which shall not be unreasonably withheld. Furthermore, SRP shall not permit any associate or secondary sponsor to suggest in any manner that its sponsorship role is as great as that of Logan's."
1,contract_0366__chunk_0013,contract_0366,exclusivity,False negative,1,0,0.9475,0.95,0,0,Pending review,,Pending review,,"patent applications and other Intellectual Property Rights related to such inventions, if any, unless otherwise expressly set forth herein. During the Term, each Party hereby provides a worldwide, exclusive, royalty free, perpetual license of such Intellectual Property Rights for use by each licensee in its business in connection with the development and marketing and commercialization of the Product. For avoidance of doubt, no Party or any of its Affiliates shall have any rights to the Background IP of the other Party nor shall any Party have rights to any trademarks, service marks, trade names, business names or product names developed by the other Party. All decisions regarding the protection and exploitation of joint investments and other Intellectual Property Rights shall be determined by the JPT or otherwise in accordance with Section 2.1. 3.7 [reserved] CERTAIN PORTIONS OF THIS EXHIBIT HAVE BEEN OMITTED PURSUANT TO A REQUEST FOR CONFIDENTIAL TREATMENT UNDER RULE 24B-2 UNDER THE SECURITIES EXCHANGE ACT OF 1934. OMISSIONS ARE DESIGNATED [***]. A COMPLETE VERSION OF THIS EXHIBIT HAS BEEN FILED SEPARATELY WITH THE SECURITIES AND EXCHANGE COMMISSION. 7"
2,contract_0195__chunk_0002,contract_0195,exclu

### Part F: Manual Review of Uncapped Liability Examples

This section reviews the six selected uncapped-liability examples,
including three false positives and three false negatives.

The review examines the contract language, transformer scores,
category-specific thresholds, and related provisions where context
is needed.

The examples show several potential issues:

- Confusion between liability caps and uncapped-liability exceptions.
- Difficulty interpreting exceptions expressed through cross-references.
- Difficulty recognizing provisions that expressly exclude certain
  obligations from liability limitations.
- Potential chunk-boundary effects when a liability cap and its
  exceptions appear in adjacent chunks.

These are preliminary assessments. Examples with unclear scope,
redacted language, or potentially incomplete annotations are marked
for further verification.

The original labels, masks, scores, and predictions are not modified.

In [100]:
# RECORD UNCAPPED LIABILITY MANUAL REVIEW

# Store the preliminary findings for each reviewed example.
# These assessments are based on the contract language inspected above.
uncapped_findings = {

    "contract_0327__chunk_0013": {
        "manual_assessment": "Ambiguous positive; requires section-level review",
        "candidate_failure_theme": "Cross-referenced liability exceptions",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The clause limits general liability but creates exceptions "
            "for claims under Section 10 and failure to pay under Section 6. "
            "The scope of these exceptions needs further review."
        )
    },

    "contract_0247__chunk_0021": {
        "manual_assessment": "Likely false negative",
        "candidate_failure_theme": "Exclusions from liability limitations",
        "label_needs_verification": "Yes",
        "review_notes": (
            "Section 9.3 expressly excludes payment obligations, early "
            "termination charges, confidentiality breaches, intellectual "
            "property misappropriation, and indemnification obligations "
            "from the liability limitations."
        )
    },

    "contract_0386__chunk_0011": {
        "manual_assessment": "Potential false negative; scope requires review",
        "candidate_failure_theme": "Uncapped liability expressed through clarification language",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The provision states that Section 3.1 does not limit Supplier's "
            "liability for specified matters. Other relevant provisions are "
            "partially redacted and require further context."
        )
    },

    "contract_0146__chunk_0017": {
        "manual_assessment": "Likely genuine false positive",
        "candidate_failure_theme": "Confusion between liability caps and uncapped liability",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The provision expressly caps D2's liability at the total amount "
            "paid or owed by the Licensee. No uncapped-liability exception "
            "appears in the selected passage."
        )
    },

    "contract_0029__chunk_0071": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Broad liability exceptions by cross-reference",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The clause establishes a general cap but excludes claims "
            "associated with several referenced sections and paragraphs. "
            "The underlying provisions should be checked before confirming "
            "whether the label is appropriate."
        )
    },

    "contract_0112__chunk_0020": {
        "manual_assessment": "Likely chunk-level false positive",
        "candidate_failure_theme": "Adjacent liability cap and exception provisions",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The selected chunk contains a maximum aggregate liability cap. "
            "The following chunk contains a separate exceptions provision. "
            "This may explain the nearby high-scoring predictions, but the "
            "selected chunk must be evaluated on its own text."
        )
    }
}


# Apply the manual findings to the uncapped-liability examples.
for chunk_id, findings in uncapped_findings.items():

    # Locate the corresponding example in the manual review table.
    row_index = manual_review.index[
        manual_review["chunk_id"] == chunk_id
    ]

    # Update only the manual-review fields.
    # Model predictions and ground-truth labels remain unchanged.
    for column, value in findings.items():
        manual_review.loc[row_index, column] = value


# Create a separate table containing the six uncapped-liability examples.
uncapped_review = manual_review[
    manual_review["category_id"] == "uncapped_liability"
].copy()


# Display the manual review results.
print("UNCAPPED LIABILITY MANUAL REVIEW")
print("Examples reviewed:", len(uncapped_review))

print("\nAssessment counts:")
print(uncapped_review["manual_assessment"].value_counts())

print("\nLabel verification:")
print(uncapped_review["label_needs_verification"].value_counts())


# Display the assessment fields and contract text.
review_columns = [
    "chunk_id",
    "error_type",
    "transformer_score",
    "transformer_threshold",
    "manual_assessment",
    "candidate_failure_theme",
    "label_needs_verification",
    "review_notes",
    "chunk_text"
]

with pd.option_context("display.max_colwidth", None):
    display(uncapped_review[review_columns])

UNCAPPED LIABILITY MANUAL REVIEW
Examples reviewed: 6

Assessment counts:
manual_assessment
Ambiguous positive; requires section-level review    1
Likely false negative                                1
Potential false negative; scope requires review      1
Likely genuine false positive                        1
Potential annotation disagreement                    1
Likely chunk-level false positive                    1
Name: count, dtype: int64

Label verification:
label_needs_verification
Yes                                  4
No apparent issue in this excerpt    2
Name: count, dtype: int64


,chunk_id,error_type,transformer_score,transformer_threshold,manual_assessment,candidate_failure_theme,label_needs_verification,review_notes,chunk_text
12,contract_0327__chunk_0013,False negative,0.8560,0.9,Ambiguous positive; requires section-level review,Cross-referenced liability exceptions,Yes,The clause limits general liability but creates exceptions for claims under Section 10 and failure to pay under Section 6. The scope of these exceptions needs further review.,"NEGLIGENCE) ARISING OUT OF OR IN CONNECTION WITH THIS AGREEMENT, EVEN IF THE PARTIES ARE AWARE OF THE POSSIBILITY OF SUCH DAMAGES.\n\nEXCEPT IN THE EVENT OF A CLAIM UNDER SECTION 10 OR FAILURE TO PAY UNDER SECTION 6, IN NO EVENT SHALL EITHER PARTY BE LIABLE TO THE\n\n 8\n\n OTHER PARTY IN AN AMOUNT GREATER THAN THE AMOUNT BOXLOT ACTUALLY PAYS TO THEGLOBE HEREUNDER.\n\n12. GENERAL.\n\n 12.1 Compliance with Laws. At its own expense, Boxlot shall comply with all applicable laws and regulations regarding providing and operating the Service. Without limiting the foregoing, Boxlot shall (a) obtain all governmental approval, permits and licenses, and make all governmental filings and registrations, necessary for the marketing and performance of the Service in every United States jurisdiction, (b) ensure that theglobe is not required to obtain any governmental approval, permits or licenses as a result of this Agreement, theglobe's performance hereunder or any payments made to theglobe hereunder, and (c) promptly resolve any assertions that the Service is illegal or violating a protected third party interest.\n\n 12.2 Governing Law. This Agreement will be governed and construed in accordance with the laws of the State of New York without giving effect to conflict of laws principles. Both parties submit to personal jurisdiction in New York and further agree that any cause of action arising under this Agreement shall be brought in a court in New York City, NY.\n\n 12.3 Publicity. Prior to the release of any press releases or other similar promotional materials related to this Agreement, the releasing party shall submit a written request for approval to the other party with a copy of the materials to be released, which request shall be made no less than 3 business days prior to the requested release date. A party shall not unreasonably withhold or delay the granting of its approval of such materials. The parties shall issue the first press release jointly.\n\n 12.4 Independent Contractors. The parties are independent contractors, and no agency, partnership, franchise, joint venture or employment relationship is intended or created by this Agreement. Neither party shall make any warranties or representations on behalf of the other party."
13,contract_0247__chunk_0021,False negative,0.5990,0.9,Likely false negative,Exclusions from liability limitations,Yes,"Section 9.3 expressly excludes payment obligations, early termination charges, confidentiality breaches, intellectual property misappropriation, and indemnification obligations from the liability limitations.","9.3 EXCLUSIONS. CHANNEL PARTNER'S PAYMENT OBLIGATIONS, LIABILITY FOR EARLY TERMINATION FEES OR CHARGES, BREACHES OF CONFIDENTIALITY BY EITHER PARTY, MISAPPROPRIATION OF INTELLECTUAL PROPERTY RIGHTS OF THE OTHER PARTY, AND THE PARTIES' INDEMNIFICATION OBLIGATIONS UNDER THIS AGREEMENT ARE EXCLUDED FROM THESE LIMITATIONS OF LIABILITY. 10. TERM AND TERMINATION 10.1 Term. This Agreement shall commence on the Effective Date and shall continue for a period of twelve (12) full calendar months (""Initial Term""). The Agreement shall automatically renew for successive one (1) year terms (each a ""Renewal Term"") unless either party provides the other party written notification of its intent to terminate the Agreement no later than sixty (60) days prior to the end of the then applicable term. Except in instances of termination of this Agreement in accordance with Sections 10.2 and 10.3 below, this Agreement shall continue, ev

### Part G: Manual Review of License Grant Examples

This section reviews the six selected license-grant examples,
including three false positives and three false negatives.

The review examines whether the contract text contains an express
grant of rights, including rights involving intellectual property,
software, trademarks, copyrights, or other licensed materials.

We also consider whether the grant is limited, conditional,
non-exclusive, or provided as a fallback when an assignment
of intellectual property is not possible.

The purpose is to distinguish genuine model errors from possible
annotation disagreements.

Preliminary review themes include:

- Explicit licensing language receiving scores below the threshold.
- Conditional or fallback license provisions.
- Limited trademark and intellectual property permissions.
- Difficulty distinguishing different types of contractual grants.

The original labels, masks, model scores, thresholds, and predictions
will remain unchanged.

In [101]:
# RECORD LICENSE GRANT MANUAL REVIEW

# Store the preliminary findings for the six license-grant examples.
license_findings = {

    "contract_0224__chunk_0001": {
        "manual_assessment": "Likely genuine false negative",
        "candidate_failure_theme": "Explicit franchise and operating rights",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The agreement expressly grants the right and license to "
            "operate an AIRSOPURE Center. It also grants exclusive rights "
            "to solicit customers within the defined territory. The "
            "transformer score of 0.8884 falls just below the 0.90 threshold."
        )
    },

    "contract_0278__chunk_0006": {
        "manual_assessment": "Likely genuine false negative",
        "candidate_failure_theme": "Explicit non-exclusive intellectual property license",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "Section 2.8 expressly states that the Licensor grants "
            "the Licensee a non-exclusive right to use trademarks, "
            "service marks, trade names, copyrights, logos, and "
            "designations during the agreement term."
        )
    },

    "contract_0130__chunk_0011": {
        "manual_assessment": "Likely false negative; inspect surrounding grant language",
        "candidate_failure_theme": "Software use, modification, distribution, and sublicensing rights",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The passage describes rights to install, copy, use, execute, "
            "modify, distribute, enhance, and sublicense software. "
            "The selected chunk begins mid-provision, so surrounding "
            "language should be checked to confirm the complete grant."
        )
    },

    "contract_0022__chunk_0021": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Conditional or fallback intellectual property license",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The passage provides an unlimited, exclusive, irrevocable, "
            "transferable, sublicensable, worldwide, perpetual, royalty-free "
            "license when certain intellectual property rights cannot "
            "be assigned. Confirm whether this conditional fallback "
            "grant qualifies under the annotation definition."
        )
    },

    "contract_0302__chunk_0100": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Conditional license grant under a terminated collaboration",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The passage states that GSK shall grant Theravance appropriate "
            "licenses under specified patents, inventions, and know-how "
            "for a terminated development collaboration product. "
            "Confirm whether the conditional nature of the grant affects "
            "the category label."
        )
    },

    "contract_0195__chunk_0004": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Limited trademark licensing permission",
        "label_needs_verification": "Yes",
        "review_notes": (
            "The passage states that SRP may license and use Logan's "
            "name and trademark in promotions directly related to the "
            "Race Car or Team. Confirm whether this limited permission "
            "falls within the license-grant annotation definition."
        )
    }
}


# Apply the findings to the corresponding license-grant examples.
for chunk_id, findings in license_findings.items():

    # Locate the example using its unique chunk identifier.
    row_index = manual_review.index[
        manual_review["chunk_id"] == chunk_id
    ]

    # Update only the manual-review fields.
    # The original model results and labels remain unchanged.
    for column, value in findings.items():
        manual_review.loc[row_index, column] = value


# Create a separate table containing the six license-grant examples.
license_review = manual_review[
    manual_review["category_id"] == "license_grant"
].copy()


# Display the review summary.
print("LICENSE GRANT MANUAL REVIEW")
print("Examples reviewed:", len(license_review))

print("\nAssessment counts:")
print(license_review["manual_assessment"].value_counts())

print("\nLabel verification:")
print(license_review["label_needs_verification"].value_counts())


# Display the contract text and review notes.
review_columns = [
    "chunk_id",
    "error_type",
    "transformer_score",
    "transformer_threshold",
    "manual_assessment",
    "candidate_failure_theme",
    "label_needs_verification",
    "review_notes",
    "chunk_text"
]

with pd.option_context("display.max_colwidth", None):
    display(license_review[review_columns])

LICENSE GRANT MANUAL REVIEW
Examples reviewed: 6

Assessment counts:
manual_assessment
Potential annotation disagreement                            3
Likely genuine false negative                                2
Likely false negative; inspect surrounding grant language    1
Name: count, dtype: int64

Label verification:
label_needs_verification
Yes                                  4
No apparent issue in this excerpt    2
Name: count, dtype: int64


,chunk_id,error_type,transformer_score,transformer_threshold,manual_assessment,candidate_failure_theme,label_needs_verification,review_notes,chunk_text
6,contract_0224__chunk_0001,False negative,0.8884,0.9,Likely genuine false negative,Explicit franchise and operating rights,No apparent issue in this excerpt,The agreement expressly grants the right and license to operate an AIRSOPURE Center. It also grants exclusive rights to solicit customers within the defined territory. The transformer score of 0.8884 falls just below the 0.90 threshold.,"1.01. AIRSOPURE grants to You, and You accept from AIRSOPURE, the right and license to operate an AIRSOPURE Center (or the ""Franchise"") for the sale or lease of AIRSOPURE's exclusive line of Products or at a location in the Exclusive Territory to be approved in writing by AIRSOPURE and listed in attached Exhibit A (the ""Exclusive Territory and Center Location""), to purchase Products from AIRSOPURE or its affiliates for resale at the Center to customers in the Exclusive Territory, and to use the Licensed Marks only in connection with the operation of the Franchise in accordance with the terms and conditions of this Agreement. AIRSOPURE grants the Franchise to You hereunder in reliance upon Your agreement to at all times operate and manage the Franchise faithfully, honestly and diligently in strict conformance with AIRSOPURE's operating procedures and specifications, as set forth herein and as otherwise from time to time communicated to You, using Your best efforts to promote and enhance the performance and operation of the Franchise.\n\n1.02. AIRSOPURE hereby grants to You the exclusive right to solicit customers for the Products by direct mail advertising, or other approved means, but not including the World Wide Web (Internet) nor by printed catalogues, in\n\n 1\n\nthe Exclusive Territory described above and in Exhibit A. Other AIRSOPURE franchisees will not be permitted to solicit customers for Products by advertising in Your Exclusive Territory. Likewise, You may not target or solicit customers for Products by advertising in other Franchisees respective Exclusive Territories. Exclusive Territories will not overlap into other Exclusive Territories. You shall not purposely solicit sales and service to customers located outside Your Exclusive Territory, provided such activity in not within an assigned Exclusive Territory. You have been granted an exclusive trade area by this Agreement.\n\n2. TERM AND RENEWAL\n\n2.01. The term of this Agreement shall be for 10 years commencing on the date of execution of this Agreement by AIRSOPURE.\n\n2.02. At the expiration of the term or any renewal term hereof, You may, at its option, renew the Franchise granted hereunder for 2 additional terms of 10 years each on the following terms and conditions:\n\n\n\n\n\nA. You shall give AIRSOPURE notice in writing of Your election to renew this Agreement at least 3 months prior to the expiration of the then-current term."
7,contract_0278__chunk_0006,False negative,0.8848,0.9,Likely genuine false negative,Explicit non-exclusive intellectual property license,No apparent issue in this excerpt,"Section 2.8 expressly states that the Licensor grants the Licensee a non-exclusive right to use trademarks, service marks, trade names, copyrights, logos, and designations during the agreement term.","Section 2.8 TRADEMARKS AND COPYRIGHT. Licensor hereby grants to Licensee a non-exclusive right to use the trademarks, service marks, trade names, copyrights, logos and designations (collectively, the ""Marks"") relating to the Licensed Products or the Documentation during the term of this Agreement in the marketing by Licensee of the Licensed Products, provided that such Marks clearly indicate Licensor as the owner of the Marks whenever the Licensed Product or Documentation is first mentioned in any written material referencing the Licensed Product and the proper symbol is used in a superscript following the Marks. Licensor 

### Part H: Manual Review of Exclusivity Examples

This section reviews the six selected exclusivity examples, including
three false positives and three false negatives.

The review examines whether the contract language establishes an
exclusive right, limits competing activities, restricts competing
products, or expressly disclaims exclusivity.

The examples show several potential patterns:

- Exclusive promotional and sponsorship rights.
- Exclusive intellectual property licensing.
- Restrictions on competing products and commercial activities.
- Difficulty interpreting explicit negation, such as "NO EXCLUSIVITY."
- Potential disagreements between the contract language and
  ground-truth annotations.

These are preliminary assessments. Examples that contain explicit
exclusivity provisions but have negative ground-truth labels will be
flagged for annotation verification.

The original labels, masks, model scores, thresholds, and predictions
will remain unchanged.

In [102]:
# RECORD EXCLUSIVITY MANUAL REVIEW

# Store the preliminary findings for the six exclusivity examples.
exclusivity_findings = {

    "contract_0195__chunk_0003": {
        "manual_assessment": "Likely genuine false negative",
        "candidate_failure_theme": "Exclusive sponsorship and advertising rights",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The passage gives Logan's exclusive use of specific Race Car "
            "sponsorship areas and restricts competing associate sponsors. "
            "The transformer score of 0.9485 is just below the 0.95 threshold."
        )
    },

    "contract_0366__chunk_0013": {
        "manual_assessment": "Likely genuine false negative",
        "candidate_failure_theme": "Exclusive intellectual property licensing",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The passage expressly provides a worldwide, exclusive, "
            "royalty-free, perpetual license of intellectual property "
            "rights for use in product development, marketing, and "
            "commercialization. The score of 0.9475 is just below "
            "the 0.95 threshold."
        )
    },

    "contract_0195__chunk_0002": {
        "manual_assessment": "Likely genuine false negative",
        "candidate_failure_theme": "Exclusive promotional rights limited to specific assets",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "The passage gives Logan's exclusive rights to promote its logo "
            "on specified uniform and driver suit areas, as well as its "
            "trademark on the driver's helmet. The score of 0.7798 is "
            "well below the 0.95 threshold."
        )
    },

    "contract_0152__chunk_0055": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Express exclusivity commitment restricting competing products",
        "label_needs_verification": "Yes",
        "review_notes": (
            "Section 3.19 is titled 'Exclusivity Commitment of SFJ' and "
            "restricts clinical trials, sales, offers for sale, and "
            "commercialization of competing products during the "
            "Exclusive Period. The negative ground-truth label should "
            "be checked against the annotation definition."
        )
    },

    "contract_0069__chunk_0011": {
        "manual_assessment": "Likely genuine false positive",
        "candidate_failure_theme": "Negation and explicit non-exclusivity",
        "label_needs_verification": "No apparent issue in this excerpt",
        "review_notes": (
            "Section 2.3 is explicitly titled 'NO RENEWAL RIGHT; "
            "NO EXCLUSIVITY.' It states that the franchisee has no "
            "exclusive territory and that BKC may develop or franchise "
            "additional restaurants nearby. This is a clear example "
            "of negated exclusivity language."
        )
    },

    "contract_0225__chunk_0040": {
        "manual_assessment": "Potential annotation disagreement",
        "candidate_failure_theme": "Express restrictions on competing product commercialization",
        "label_needs_verification": "Yes",
        "review_notes": (
            "Section 9.2 is titled 'Exclusivity' and prohibits the "
            "company and its affiliates from marketing, promoting, "
            "selling, distributing, or commercializing competing "
            "products in the territory without prior written consent. "
            "The negative ground-truth label should be verified."
        )
    }
}


# Apply the findings to the corresponding exclusivity examples.
for chunk_id, findings in exclusivity_findings.items():

    # Locate the example using its unique chunk identifier.
    row_index = manual_review.index[
        manual_review["chunk_id"] == chunk_id
    ]

    # Check that the example exists before updating its review fields.
    if len(row_index) == 0:
        print("WARNING: Example not found:", chunk_id)
        continue

    # Update only the manual-review fields.
    # Original model predictions and ground-truth labels remain unchanged.
    for column, value in findings.items():
        manual_review.loc[row_index, column] = value


# Create a separate table containing the six exclusivity examples.
exclusivity_review = manual_review[
    manual_review["category_id"] == "exclusivity"
].copy()


# Display the review summary.
print("EXCLUSIVITY MANUAL REVIEW")
print("Examples reviewed:", len(exclusivity_review))

print("\nAssessment counts:")
print(exclusivity_review["manual_assessment"].value_counts())

print("\nLabel verification:")
print(exclusivity_review["label_needs_verification"].value_counts())


# Display the contract text and review notes.
review_columns = [
    "chunk_id",
    "error_type",
    "transformer_score",
    "transformer_threshold",
    "manual_assessment",
    "candidate_failure_theme",
    "label_needs_verification",
    "review_notes",
    "chunk_text"
]

with pd.option_context("display.max_colwidth", None):
    display(exclusivity_review[review_columns])

EXCLUSIVITY MANUAL REVIEW
Examples reviewed: 6

Assessment counts:
manual_assessment
Likely genuine false negative        3
Potential annotation disagreement    2
Likely genuine false positive        1
Name: count, dtype: int64

Label verification:
label_needs_verification
No apparent issue in this excerpt    4
Yes                                  2
Name: count, dtype: int64


,chunk_id,error_type,transformer_score,transformer_threshold,manual_assessment,candidate_failure_theme,label_needs_verification,review_notes,chunk_text
0,contract_0195__chunk_0003,False negative,0.9485,0.95,Likely genuine false negative,Exclusive sponsorship and advertising rights,No apparent issue in this excerpt,The passage gives Logan's exclusive use of specific Race Car sponsorship areas and restricts competing associate sponsors. The transformer score of 0.9485 is just below the 0.95 threshold.,"(H) TRANSPORTER. SRP is attempting to purchase or lease a 53 foot semi-trailer to be pulled by a Peterbilt class eight tractor which will function as the Race Car transporter (the ""Transporter""). In the event SRP is successful in procuring the Transporter, it will be lettered and pictured with Logan's racing graphics by SRP up to a maximum cost allowance of Two Thousand Five Hundred and No/100's ($2,500.00), with Logan's retaining the right to apply more extensive graphics at its own expense if Logan's so chooses. Any associate sponsor graphics appearing on the Transporter will be displayed in a subservient manner to clearly reflect Logan's as the primary sponsor of the Team.\n\n (I) AUTHORITY OF SRP. SRP hereby warrants that it has the authority to grant all rights to Baker under this Article 3.\n\n 4. ASSOCIATE AND SECONDARY SPONSORSHIPS. SRP shall have the right to obtain secondary and associate sponsors for the Race Car and Team, subject to the approval of Logan's, which approval shall not be unreasonably withheld. SRP may not obtain any associate or secondary sponsors whose products or concepts compete with Logan's. Any associate and/or secondary sponsor may promote its role as a sponsor by signage, logos or trademarks on the Race Car, provided that such associate and/or secondary sponsors signage, logos or trademarks used on the Race Car, when viewed collectively, shall not cover any area greater than 25% of that of Logan's signage, logos or trademarks, nor conflict with the graphic design, or in other locations or on clothing as may be permitted by the terms of the Agreement. Logan's shall have the exclusive use of the hood, the rear quarter panel area above the tire and the bottom of the deck lid (rear facing panel, TV panel) of the Race Car, as shown on Exhibit C to this Agreement. All other areas permitted by NASCAR for sponsor signage, logos or trademarks and the ""spoiler space"" are reserved for NASCAR, SRP and the associate and secondary sponsors. All associate and secondary sponsors' graphic layouts for the Race Car are subject to Logan's approval, which shall not be unreasonably withheld. Furthermore, SRP shall not permit any associate or secondary sponsor to suggest in any manner that its sponsorship role is as great as that of Logan's."
1,contract_0366__chunk_0013,False negative,0.9475,0.95,Likely genuine false negative,Exclusive intellectual property licensing,No apparent issue in this excerpt,"The passage expressly provides a worldwide, exclusive, royalty-free, perpetual license of intellectual property rights for use in product development, marketing, and commercialization. The score of 0.9475 is just below the 0.95 threshold.","patent applications and other Intellectual Property Rights related to such inventions, if any, unless otherwise expressly set forth herein. During the Term, each Party hereby provides a worldwide, exclusive, royalty free, perpetual license of such Intellectual Property Rights for use by each licensee in its business in connection with the development and marketing and commercialization of the Product. For avoidance of doubt, no Party or any of its Affiliates shall have any rights to the Background IP of the other Party nor shall any Party have rights to any trademarks, service marks, trade names, business names or product names developed by the other Party. All decisions regarding the protection and exploitation of joint investments and other Intellectual Property Rights shall be det

## STEP 6: CONSOLIDATE FAILURE THEMES AND MANUAL REVIEW FINDINGS

The manual review covered 18 transformer error examples across the
three categories with the highest number of transformer errors:

- Uncapped liability
- License grant
- Exclusivity

This section consolidates the findings from the three category reviews.

The summary includes:
- Number of examples reviewed in each category.
- False-positive and false-negative counts in the review sample.
- Number of examples flagged for ground-truth label verification.
- Candidate failure themes identified during manual inspection.

The findings are based on a targeted sample of transformer errors,
not a review of every incorrect prediction.

Potential annotation disagreements are documented separately from
confirmed model errors. No ground-truth labels or model predictions
are changed in this analysis.

In [103]:
# CONSOLIDATE FAILURE THEMES

# Group the 18 manually reviewed examples by contract category.
# Summarize the review findings for each category.
failure_theme_summary = manual_review.groupby("category_id").agg(

    # Count the number of reviewed examples.
    examples_reviewed=("chunk_id", "size"),

    # Count false positives in the review sample.
    false_positives=(
        "error_type",
        lambda values: (values == "False positive").sum()
    ),

    # Count false negatives in the review sample.
    false_negatives=(
        "error_type",
        lambda values: (values == "False negative").sum()
    ),

    # Count examples flagged for ground-truth verification.
    labels_flagged=(
        "label_needs_verification",
        lambda values: (values == "Yes").sum()
    ),

    # Combine the candidate themes identified during manual review.
    candidate_failure_themes=(
        "candidate_failure_theme",
        lambda values: "; ".join(
            sorted(set(values.dropna()) - {""})
        )
    )

).reset_index()

# SUMMARIZE MANUAL ASSESSMENTS

# Count the different manual assessment decisions across all examples.
assessment_summary = (
    manual_review["manual_assessment"]
    .value_counts()
    .reset_index()
)

assessment_summary.columns = [
    "manual_assessment",
    "examples"
]

# VALIDATE THE CONSOLIDATED REVIEW

# Check that all 18 selected examples are included.
print("TOTAL EXAMPLES REVIEWED:", len(manual_review))

# Check that all three selected categories are represented.
print(
    "CATEGORIES REVIEWED:",
    manual_review["category_id"].nunique()
)

# Confirm that no example is still awaiting manual assessment.
print(
    "All assessments completed:",
    manual_review["manual_assessment"].ne("Pending review").all()
)

# Confirm that every example has a candidate failure theme.
print(
    "All examples have a candidate theme:",
    manual_review["candidate_failure_theme"].ne("").all()
)

# Confirm that the original labels and predictions were not changed.
original_results = transformer_error_review[
    [
        "chunk_id",
        "category_id",
        "label",
        "transformer_prediction",
        "transformer_score",
        "transformer_threshold"
    ]
]

updated_results = manual_review[
    [
        "chunk_id",
        "category_id",
        "label",
        "transformer_prediction",
        "transformer_score",
        "transformer_threshold"
    ]
]

integrity_check = original_results.merge(
    updated_results,
    on=["chunk_id", "category_id"],
    suffixes=("_original", "_review"),
    validate="one_to_one"
)

print(
    "Original labels preserved:",
    (
        integrity_check["label_original"]
        == integrity_check["label_review"]
    ).all()
)

print(
    "Original predictions preserved:",
    (
        integrity_check["transformer_prediction_original"]
        == integrity_check["transformer_prediction_review"]
    ).all()
)

print(
    "Original scores preserved:",
    np.isclose(
        integrity_check["transformer_score_original"],
        integrity_check["transformer_score_review"]
    ).all()
)


# DISPLAY THE CONSOLIDATED FINDINGS

print("\nCONSOLIDATED FAILURE THEME SUMMARY")
display(failure_theme_summary)

print("\nMANUAL ASSESSMENT SUMMARY")
display(assessment_summary)

TOTAL EXAMPLES REVIEWED: 18
CATEGORIES REVIEWED: 3
All assessments completed: True
All examples have a candidate theme: True
Original labels preserved: True
Original predictions preserved: True
Original scores preserved: True

CONSOLIDATED FAILURE THEME SUMMARY


,category_id,examples_reviewed,false_positives,false_negatives,labels_flagged,candidate_failure_themes
0,exclusivity,6,3,3,2,Exclusive intellectual property licensing; Exc...
1,license_grant,6,3,3,4,Conditional license grant under a terminated c...
2,uncapped_liability,6,3,3,4,Adjacent liability cap and exception provision...



MANUAL ASSESSMENT SUMMARY


,manual_assessment,examples
0,Potential annotation disagreement,6
1,Likely genuine false negative,5
2,Likely genuine false positive,2
3,Likely false negative; inspect surrounding gra...,1
4,Ambiguous positive; requires section-level review,1
5,Likely false negative,1
6,Potential false negative; scope requires review,1
7,Likely chunk-level false positive,1


## STEP 7: EXPORT FINAL RESULTS

This section saves the completed analysis tables and validation
comparison dataset in the november_task11_outputs folder.

The final report below summarizes the findings, limitations,
and overall conclusions.

In [105]:
# EXPORT FINAL RESULTS

# Create the output folder.
OUTPUT_DIR = PROJECT_ROOT / "november_task11_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remove an old README from the output folder.
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    readme_path.unlink()

# Define the CSV files to export.
csv_exports = {
    "task11_overall_outcome_summary.csv": outcome_summary,
    "task11_lr_category_summary.csv": lr_category_summary,
    "task11_keyword_category_summary.csv": keyword_category_summary,
    "task11_transformer_error_summary.csv": transformer_error_summary,
    "task11_manual_error_review.csv": manual_review,
    "task11_failure_theme_summary.csv": failure_theme_summary,
    "task11_manual_assessment_summary.csv": assessment_summary,
    "task11_transformer_false_positives.csv": transformer_false_positives,
    "task11_transformer_false_negatives.csv": transformer_false_negatives
}

# Export each analysis table.
for filename, dataframe in csv_exports.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)

# Export the complete eligible comparison dataset.
error_comparison.to_parquet(
    OUTPUT_DIR / "task11_eligible_error_comparison.parquet",
    index=False
)

# Confirm the exported files.
print("OUTPUT FOLDER:", OUTPUT_DIR)
print("\nEXPORTED FILES:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

print("\nCSV exports complete:", all(
    (OUTPUT_DIR / filename).exists()
    for filename in csv_exports
))

print(
    "Parquet export complete:",
    (OUTPUT_DIR / "task11_eligible_error_comparison.parquet").exists()
)

OUTPUT FOLDER: C:\Users\DKola\Accenture-1O-contract-review-challenge\notebooks\november_task11_outputs

EXPORTED FILES:
task11_eligible_error_comparison.parquet
task11_failure_theme_summary.csv
task11_keyword_category_summary.csv
task11_lr_category_summary.csv
task11_manual_assessment_summary.csv
task11_manual_error_review.csv
task11_overall_outcome_summary.csv
task11_transformer_error_summary.csv
task11_transformer_false_negatives.csv
task11_transformer_false_positives.csv

CSV exports complete: True
Parquet export complete: True


# FINAL REPORT: NOVEMBER TASK 11

## 1. Findings

The transformer was compared against the logistic regression and keyword
baselines using 28,690 eligible validation chunk-category pairs.

**Transformer vs. Logistic Regression**
- Newly corrected errors: 396
- New errors introduced: 91
- Persistent errors: 114
- Net reduction in errors: 305

**Transformer vs. Keyword Baseline**
- Newly corrected errors: 446
- New errors introduced: 106
- Persistent errors: 99
- Net reduction in errors: 340

The transformer produced 205 incorrect predictions:
- False positives: 110
- False negatives: 95

The three categories with the highest transformer error counts were:
- Uncapped liability: 33 errors
- License grant: 32 errors
- Exclusivity: 23 errors

Manual review covered 18 selected examples across these categories.
The review identified potential patterns involving liability exceptions,
explicit licensing provisions, exclusivity restrictions, and negation.

Ten examples were flagged for further ground-truth annotation verification.

## 2. Limitations

- The analysis uses the available validation split and does not establish
  performance on unseen contracts.
- Forty masked chunk-category pairs were excluded.
- The supplied transformer JSON identifies its model version as
  `cuad_model_v1`, but does not independently authenticate the model checkpoint.
- All 28,730 JSON thresholds matched Task 8 Seed 42 thresholds within
  the specified numerical tolerance. This supports threshold consistency,
  not complete model provenance.
- The original weighted Legal-BERT score files were not available in
  the searched project locations. Existing JSON predictions were used.
- Task 7 produced keyword and logistic regression outputs. A TF-IDF
  prediction output was not available, so no TF-IDF comparison is reported.
- Manual review covered 18 selected errors, not every transformer error.
- Potential annotation disagreements require verification against the
  project's annotation guidelines. No ground-truth labels were changed.
- This analysis evaluates model predictions and is not a legal assessment
  of the underlying contracts.

## 3. Summary

The transformer corrected more errors than it introduced when compared
with both baselines on the eligible validation data.

The largest overall improvements were observed against the keyword
baseline, while uncapped liability, license grant, and exclusivity
were identified as priority areas for further error analysis.

The manual review identified potential model failure patterns and
examples requiring annotation verification.

All comparisons reused existing predictions and frozen thresholds.
No model retraining, threshold adjustment, or ground-truth label
modification was performed.